<a href="https://colab.research.google.com/github/pronabpaul/neuroimaging-dashboard/blob/main/Final_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **How to run the dashboard**

1. **Run the cell** in this notebook.  
  

2. **Click on the public URL** that appears at the bottom of the output.

3. **Use the dashboard** - upload a brain slice or image click on a region, and adjust parameters to segment it.  
   Export the mask or overlay when done.

In [ ]:
# %% [markdown]
## Neuroimaging Dashboard - Final Version

!pip install -q gradio scipy scikit-image

# %%
import gradio as gr
import numpy as np
import cv2
import heapq
from scipy import ndimage
from skimage.segmentation import active_contour, morphological_chan_vese
from skimage import measure, filters
from skimage.morphology import remove_small_objects, disk
import traceback
import io
import base64
from PIL import Image

# =================
# Dark blue theme
# =================
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
body, .gradio-container { font-family: 'Inter', sans-serif; background: #0b0f19; }
.gradio-container { max-width: 100% !important; padding: 1rem 1.5rem; }
.main-header { text-align: center; padding: 0.75rem 0 0.25rem; border-bottom: 1px solid #1e2a3a; margin-bottom: 1rem; }
.main-header h1 { color: #c4d5f5; font-size: 1.8rem; font-weight: 600; margin: 0; }
.main-header p { color: #7d8ba0; font-size: 0.85rem; margin-top: 0.15rem; }
.card { background: #111827; border: 1px solid #1e2a3a; border-radius: 14px; padding: 1rem; box-shadow: 0 4px 12px rgba(0,0,0,0.5); margin-bottom: 1rem; }
.gr-button { background: #1e3a5f !important; color: #d0e0ff !important; border: 1px solid #2a4a7f !important; border-radius: 8px !important; font-weight: 500 !important; transition: 0.2s; }
.gr-button:hover { background: #244a7a !important; }
.gr-slider input[type=range] { accent-color: #3b82f6; }
.gr-checkbox input[type=checkbox]:checked { accent-color: #3b82f6; }
.gr-radio input[type=radio]:checked { accent-color: #3b82f6; }
.analytics-header { background: #1e3a5f; color: #d0e0ff; padding: 0.5rem 1rem; border-radius: 10px 10px 0 0; font-weight: 500; }
.analytics-body { background: #111827; border: 1px solid #1e2a3a; border-top: none; border-radius: 0 0 10px 10px; padding: 1rem; min-height: 300px; }

/* Force white text on all markdown elements inside analytics-body */
.analytics-body,
.analytics-body *,
.analytics-body .prose,
.analytics-body .prose *,
.analytics-body p,
.analytics-body li,
.analytics-body strong,
.analytics-body em,
.analytics-body h1,
.analytics-body h2,
.analytics-body h3,
.analytics-body h4,
.analytics-body code {
    color: #ffffff !important;
}
"""

# ===============
# Synthetic brain
# ===============
def generate_synthetic_brain(size=(512, 512)):
    h, w = size
    cx, cy = w // 2, h // 2
    img = np.full((h, w), 30, dtype=np.uint8)
    cv2.ellipse(img, (cx, cy), (w // 3, h * 2 // 5), 0, 0, 360, 110, -1)
    bright_region_offset_y = h // 8
    cv2.ellipse(img, (cx, cy - bright_region_offset_y),
                (w // 13, h // 17), 0, 0, 360, 220, -1)
    rng = np.random.RandomState(42)
    noise = rng.normal(0, 5, (h, w)).astype(np.float32)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return img

# =========================================
# Fast local variance via box filter (O(N))
# ==========================================
def fast_local_variance(img_gray, window=5):
    kernel = np.ones((window, window), dtype=np.float32) / (window*window)
    mean = cv2.filter2D(img_gray.astype(np.float32), -1, kernel, borderType=cv2.BORDER_REFLECT)
    sq_mean = cv2.filter2D(img_gray.astype(np.float32)**2, -1, kernel, borderType=cv2.BORDER_REFLECT)
    var = sq_mean - mean**2
    var[var < 0] = 0
    return var

# =======================
# Preprocessing pipeline
# =======================
def preprocess_image(img_gray, apply_clahe, apply_bilateral, apply_aniso):
    out = img_gray.copy()
    if apply_aniso:
        out = anisotropic_diffusion(out, niter=5, kappa=30, gamma=0.1)
    if apply_bilateral:
        out = cv2.bilateralFilter(out, d=7, sigmaColor=40, sigmaSpace=40)
    if apply_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        out = clahe.apply(out)
    return out

def anisotropic_diffusion(img, niter=5, kappa=30, gamma=0.1):
    out = img.astype(np.float32)
    for _ in range(niter):
        padded = cv2.copyMakeBorder(out, 1,1,1,1, cv2.BORDER_REFLECT)
        n = padded[0:-2, 1:-1] - out
        s = padded[2:,   1:-1] - out
        e = padded[1:-1, 2:]   - out
        w = padded[1:-1, 0:-2] - out
        ne = padded[0:-2, 2:]  - out
        nw = padded[0:-2, 0:-2]- out
        se = padded[2:,   2:]  - out
        sw = padded[2:,   0:-2]- out
        cn = 1.0 / (1.0 + (n/kappa)**2)
        cs = 1.0 / (1.0 + (s/kappa)**2)
        ce = 1.0 / (1.0 + (e/kappa)**2)
        cw = 1.0 / (1.0 + (w/kappa)**2)
        cne = 1.0 / (1.0 + (ne/kappa)**2)
        cnw = 1.0 / (1.0 + (nw/kappa)**2)
        cse = 1.0 / (1.0 + (se/kappa)**2)
        csw = 1.0 / (1.0 + (sw/kappa)**2)
        out += gamma * (cn*n + cs*s + ce*e + cw*w +
                        0.5*(cne*ne + cnw*nw + cse*se + csw*sw))
    return np.clip(out, 0, 255).astype(np.uint8)

# =========================
# Multi-scale gradient
# =========================
def multi_scale_gradient(img, use_multi_scale=False):
    gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    if use_multi_scale:
        gx2 = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=5)
        gy2 = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=5)
        mag2 = cv2.magnitude(gx2, gy2)
        mag = 0.7*mag + 0.3*mag2
    return cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX)

# ==========================================================
# Seed refinement - using preprocessed image consistently
# ==========================================================
def refine_seed(img, seed, radius=3, texture_map=None):
    x, y = seed
    h, w = img.shape
    ymin, ymax = max(0, y-radius), min(h, y+radius+1)
    xmin, xmax = max(0, x-radius), min(w, x+radius+1)
    local = img[ymin:ymax, xmin:xmax].astype(np.float32)
    if texture_map is not None:
        local_tex = texture_map[ymin:ymax, xmin:xmax]
        best_idx = np.unravel_index(np.argmin(local_tex), local_tex.shape)
    else:
        grad = ndimage.generic_gradient_magnitude(local, ndimage.sobel)
        best_idx = np.unravel_index(np.argmin(grad), grad.shape)
    new_x = int(xmin + best_idx[1])
    new_y = int(ymin + best_idx[0])
    return (new_x, new_y)

# =================
# Fast LBP image
# ==================
def compute_lbp_image(img_gray):
    h, w = img_gray.shape
    lbp = np.zeros((h, w), dtype=np.int32)
    padded = cv2.copyMakeBorder(img_gray, 1,1,1,1, cv2.BORDER_REFLECT)
    neighbours = [(-1,-1), (-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1)]
    for i, (dy, dx) in enumerate(neighbours):
        shifted = padded[1+dy:1+dy+h, 1+dx:1+dx+w]
        lbp |= ((shifted >= img_gray).astype(np.int32) << i)
    return lbp.astype(np.uint8)

# =================================
# Fast local entropy using skimage
# =================================
def fast_local_entropy(img_gray, radius=2):
    selem = disk(radius)
    entropy_map = filters.rank.entropy(img_gray, selem)
    return entropy_map.astype(np.float32)

# ========================
# Intensity normalization
# ========================
def standardize_image(img_gray):
    mean = np.mean(img_gray)
    std = np.std(img_gray) + 1e-8
    return (img_gray - mean) / std

# ==================
# Region growing
# ==================
def comprehensive_region_growing(img_gray, seed, fixed_delta, adaptive_thresh, sensitivity,
                                 use_dynamic_stats, use_multi_scale, use_texture, texture_weight,
                                 use_gradient_dir, grad_dir_weight,
                                 gradient_weight, distance_weight, edge_factor=0.5,
                                 use_cost_accept=False, cost_sensitivity=2.0,
                                 use_statistical=False, stat_threshold=3.0,
                                 preprocessed_img=None,
                                 output_similarity=False):
    h, w = img_gray.shape
    diag = np.sqrt(h*h + w*w) + 1e-6
    x, y = seed
    if not (0 <= x < w and 0 <= y < h):
        return (np.zeros((h, w), dtype=bool), None)

    work_img = preprocessed_img if preprocessed_img is not None else img_gray
    seed_intensity = int(work_img[y, x])

    grad_norm = multi_scale_gradient(work_img, use_multi_scale)
    grad_u8 = (grad_norm * 255).astype(np.uint8)
    thresh_val, _ = cv2.threshold(grad_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edge_thresh = max(1, int(thresh_val * edge_factor))

    # Statistical model initialization
    if use_statistical:
        normalized_img = standardize_image(work_img)
        lbp_img = compute_lbp_image(work_img).astype(np.float32)
        var_map = fast_local_variance(work_img, 5)
        entropy_map = fast_local_entropy(work_img, radius=2)
        grad_map = grad_norm
        num_features = 5
        seed_features = np.array([
            normalized_img[y, x],
            var_map[y, x],
            grad_map[y, x],
            lbp_img[y, x],
            entropy_map[y, x]
        ])
        feature_count = 1
        feature_mean = seed_features.copy()
        feature_cov = np.eye(num_features) * 1e-2
        cov_sum = np.zeros((num_features, num_features))
        inv_cov = np.linalg.inv(feature_cov + np.eye(num_features)*1e-3)
    else:
        num_features = 0
        feature_mean = feature_cov = inv_cov = None
        normalized_img = lbp_img = var_map = entropy_map = None

    texture_map = fast_local_variance(work_img, 5) if use_texture else None

    # Delta calculation
    if adaptive_thresh:
        half = 5
        patch = work_img[max(0, y-half):min(h, y+half+1), max(0, x-half):min(w, x+half+1)].astype(np.float32)
        med = np.median(patch)
        mad = np.median(np.abs(patch - med)) + 1e-6
        delta = max(1, int(sensitivity * mad))
    else:
        delta = fixed_delta

    # Initialize growing statistics
    region_count = 1
    region_sum = float(seed_intensity)
    region_mean = seed_intensity

    tex_count = 1
    tex_mean = float(texture_map[y, x]) if texture_map is not None else 0.0
    tex_M2 = 0.0
    tex_std = 0.0

    # Gradient direction: circular mean with magnitude weights + guard
    if use_gradient_dir and grad_dir_weight > 0:
        grad_x = cv2.Sobel(work_img, cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(work_img, cv2.CV_32F, 0, 1, ksize=3)
        grad_mag_dir = cv2.magnitude(grad_x, grad_y)
        grad_mag_dir_norm = cv2.normalize(grad_mag_dir, None, 0, 1, cv2.NORM_MINMAX)
        max_mag = np.max(grad_mag_dir_norm)
        seed_mag = grad_mag_dir_norm[y, x]
        if max_mag < 1e-8 or seed_mag < 0.01 * max_mag:
            dir_weight = 0.0
            direction = None
            grad_mag_dir_norm = None
            ref_dir = 0.0
            dir_total_weight = 0.0
            dir_sum_sin = 0.0
            dir_sum_cos = 0.0
        else:
            direction = np.arctan2(grad_y, grad_x)
            ref_dir = direction[y, x]
            w = seed_mag
            dir_total_weight = w
            dir_sum_sin = np.sin(ref_dir) * w
            dir_sum_cos = np.cos(ref_dir) * w
            dir_weight = grad_dir_weight
    else:
        direction = None
        grad_mag_dir_norm = None
        ref_dir = 0.0
        dir_weight = 0.0
        dir_total_weight = 0.0
        dir_sum_sin = 0.0
        dir_sum_cos = 0.0

    cost_count = 0
    cost_mean = 0.0
    cost_M2 = 0.0

    heap = []
    mask = np.zeros((h, w), dtype=bool)
    mask[y, x] = True
    similarity_map = np.full((h, w), np.inf, dtype=np.float32) if output_similarity else None
    if similarity_map is not None:
        similarity_map[y, x] = 0.0

    if not use_statistical:
        cum_cost_map = np.full((h, w), np.inf, dtype=np.float64)
        cum_cost_map[y, x] = 0.0
    else:
        cum_cost_map = None

    neighbours = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]

    def angle_diff(a, b):
        diff = np.abs(a - b)
        return np.minimum(diff, 2*np.pi - diff)

    def mahalanobis_distance(f_vec, mean, inv_cov_mat):
        diff = f_vec - mean
        return np.sqrt(np.dot(np.dot(diff, inv_cov_mat), diff))

    # --- Statistical Growing (Mahalanobis) ---
    if use_statistical:
        for dx, dy in neighbours:
            nx, ny = x+dx, y+dy
            if 0 <= nx < w and 0 <= ny < h:
                f_vec = np.array([
                    normalized_img[ny, nx],
                    var_map[ny, nx],
                    grad_map[ny, nx],
                    lbp_img[ny, nx],
                    entropy_map[ny, nx]
                ])
                dist = mahalanobis_distance(f_vec, feature_mean, inv_cov)
                heapq.heappush(heap, (dist, nx, ny, f_vec))
        while heap:
            dist, cx, cy, f_vec = heapq.heappop(heap)
            if mask[cy, cx]:
                continue
            if grad_u8[cy, cx] > edge_thresh:
                continue
            fresh_dist = mahalanobis_distance(f_vec, feature_mean, inv_cov)
            if feature_count < 20:
                accept = fresh_dist <= stat_threshold * 2.0
            else:
                accept = fresh_dist <= stat_threshold
            if accept:
                mask[cy, cx] = True
                if similarity_map is not None:
                    similarity_map[cy, cx] = fresh_dist
                feature_count += 1
                delta_vec = f_vec - feature_mean
                feature_mean += delta_vec / feature_count
                delta2_vec = f_vec - feature_mean
                cov_sum += np.outer(delta_vec, delta2_vec)
                feature_cov = cov_sum / (feature_count - 1) if feature_count > 1 else np.eye(num_features)*1e-2
                reg_cov = feature_cov + np.eye(num_features) * 1e-3
                try:
                    inv_cov = np.linalg.inv(reg_cov)
                except np.linalg.LinAlgError:
                    inv_cov = np.eye(num_features) * 1e3
                region_count += 1
                region_sum += work_img[cy, cx]
                region_mean = region_sum / region_count
                for dx, dy in neighbours:
                    nx, ny = cx+dx, cy+dy
                    if 0 <= nx < w and 0 <= ny < h and not mask[ny, nx]:
                        f_vec_new = np.array([
                            normalized_img[ny, nx],
                            var_map[ny, nx],
                            grad_map[ny, nx],
                            lbp_img[ny, nx],
                            entropy_map[ny, nx]
                        ])
                        d_new = mahalanobis_distance(f_vec_new, feature_mean, inv_cov)
                        heapq.heappush(heap, (d_new, nx, ny, f_vec_new))
    # ---- Classic growing - Dijkstra-optimal cumulative cost + circular direction ---
    else:
        def calc_local_cost(v, tex_val, grad, eucl_d, ref_intensity, tex_curr_mean, tex_curr_std, nx, ny):
            int_cost = min(1.0, abs(v - ref_intensity) / max(delta, 1))
            grad_cost = grad
            if texture_map is not None and use_texture and tex_curr_std > 1e-8:
                tex_cost = min(1.0, abs(tex_val - tex_curr_mean) / (tex_curr_std * 2.0 + 1e-8))
            else:
                tex_cost = 0.0
            dist_cost = min(1.0, eucl_d / diag)
            dir_cost = 0.0
            if direction is not None and grad_mag_dir_norm is not None and dir_total_weight > 0:
                cur_dir = np.arctan2(dir_sum_sin, dir_sum_cos)
                dir_cost = angle_diff(direction[ny, nx], cur_dir) / np.pi
            total = (int_cost +
                     gradient_weight * grad_cost +
                     texture_weight * tex_cost +
                     distance_weight * dist_cost +
                     dir_weight * dir_cost)
            return total / (1.0 + gradient_weight + texture_weight + distance_weight + dir_weight)

        for dx, dy in neighbours:
            nx, ny = x+dx, y+dy
            if 0 <= nx < w and 0 <= ny < h:
                v = float(work_img[ny, nx])
                tex = texture_map[ny, nx] if texture_map is not None else 0.0
                grad = grad_norm[ny, nx]
                local_cost = calc_local_cost(v, tex, grad, np.sqrt((nx-x)**2 + (ny-y)**2),
                                             seed_intensity, tex_mean, tex_std, nx, ny)
                cum_cost = local_cost
                cum_cost_map[ny, nx] = cum_cost
                heapq.heappush(heap, (cum_cost, nx, ny, v, tex, grad, np.sqrt((nx-x)**2 + (ny-y)**2), local_cost))

        while heap:
            cum_cost, cx, cy, cval, ctex, cgrad, ceucl, local_cost = heapq.heappop(heap)
            if mask[cy, cx]:
                continue
            if cum_cost > cum_cost_map[cy, cx] + 1e-12:
                continue
            if grad_u8[cy, cx] > edge_thresh:
                continue

            if use_cost_accept:
                if cost_count < 20:
                    accept = local_cost <= 0.5
                else:
                    cost_std = np.sqrt(max(0, cost_M2 / (cost_count - 1)))
                    threshold = cost_mean + cost_sensitivity * cost_std
                    accept = local_cost <= threshold
            else:
                ref_intensity = region_mean if use_dynamic_stats else seed_intensity
                accept = abs(cval - ref_intensity) <= delta

            if accept and texture_map is not None and use_texture and tex_count > 5:
                if tex_count > 1:
                    tex_std = np.sqrt(max(0, tex_M2 / (tex_count - 1)))
                else:
                    tex_std = 0.0
                if abs(ctex - tex_mean) > 2.5 * max(tex_std, 1e-8):
                    accept = False

            if accept and direction is not None and dir_weight > 0 and dir_total_weight > 0:
                cur_dir = np.arctan2(dir_sum_sin, dir_sum_cos)
                if angle_diff(direction[cy, cx], cur_dir) > np.pi/4:
                    accept = False

            if accept:
                mask[cy, cx] = True
                if similarity_map is not None:
                    similarity_map[cy, cx] = local_cost

                region_count += 1
                region_sum += cval
                region_mean = region_sum / region_count

                if direction is not None and dir_weight > 0 and grad_mag_dir_norm is not None:
                    ang = direction[cy, cx]
                    w = grad_mag_dir_norm[cy, cx] + 1e-6
                    dir_sum_sin += np.sin(ang) * w
                    dir_sum_cos += np.cos(ang) * w
                    dir_total_weight += w

                if texture_map is not None and use_texture:
                    tex_count += 1
                    delta_tex = ctex - tex_mean
                    tex_mean += delta_tex / tex_count
                    delta2 = ctex - tex_mean
                    tex_M2 += delta_tex * delta2

                if use_cost_accept:
                    cost_count += 1
                    delta_cost = local_cost - cost_mean
                    cost_mean += delta_cost / cost_count
                    delta2_cost = local_cost - cost_mean
                    cost_M2 += delta_cost * delta2_cost

                for dx, dy in neighbours:
                    nx, ny = cx+dx, cy+dy
                    if 0 <= nx < w and 0 <= ny < h and not mask[ny, nx]:
                        v = float(work_img[ny, nx])
                        tex = texture_map[ny, nx] if texture_map is not None else 0.0
                        grad = grad_norm[ny, nx]
                        eucl_new = np.sqrt((nx-x)**2 + (ny-y)**2)
                        if texture_map is not None and use_texture and tex_count > 1:
                            curr_tex_std = np.sqrt(max(0, tex_M2 / (tex_count - 1)))
                        else:
                            curr_tex_std = 0.0
                        new_local_cost = calc_local_cost(v, tex, grad, eucl_new,
                                                         region_mean if use_dynamic_stats else seed_intensity,
                                                         tex_mean, curr_tex_std, nx, ny)
                        new_cum_cost = cum_cost + new_local_cost
                        if new_cum_cost < cum_cost_map[ny, nx]:
                            cum_cost_map[ny, nx] = new_cum_cost
                            heapq.heappush(heap, (new_cum_cost, nx, ny, v, tex, grad, eucl_new, new_local_cost))

    # Post-processing
    num_labels, labels, _, _ = cv2.connectedComponentsWithStats((mask.astype(np.uint8))*255, connectivity=8)
    seed_label = labels[y, x]
    if seed_label > 0:
        clean = (labels == seed_label)
    else:
        clean = mask.copy()
    filled = ndimage.binary_fill_holes(clean)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    u8 = (filled.astype(np.uint8))*255
    closed = cv2.morphologyEx(u8, cv2.MORPH_CLOSE, kernel)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)
    clean = opened > 0
    min_obj = max(50, int(0.001 * h * w))
    clean = remove_small_objects(clean, min_size=min_obj)
    if not clean[y, x]:
        clean = (labels == seed_label)
    return (clean, similarity_map if output_similarity else None)

# =================
# Boundary snapping
# =================
def refine_boundary_snakes(img_gray, mask, iterations=None):
    if np.sum(mask) == 0:
        return mask
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return mask
    cnt = max(contours, key=cv2.contourArea)
    if cnt.shape[0] < 4:
        return mask
    length = cv2.arcLength(cnt, True)
    if iterations is None:
        iterations = max(10, min(50, int(length / 5)))
    snake = cnt.astype(np.float32).reshape(-1, 2)[:, ::-1]
    dists = np.linalg.norm(np.diff(snake, axis=0), axis=1)
    cum = np.insert(np.cumsum(dists), 0, 0)
    total = cum[-1]
    npts = min(100, snake.shape[0])
    samples = np.linspace(0, total, npts)
    snake = np.array([np.interp(samples, cum, snake[:, i]) for i in range(2)]).T
    grad_x = cv2.Sobel(img_gray.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img_gray.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = cv2.magnitude(grad_x, grad_y)
    grad_mag = cv2.GaussianBlur(grad_mag, (5,5), 0)
    grad_mag = cv2.normalize(grad_mag, None, 0, 1, cv2.NORM_MINMAX)
    edge_map = 1.0 - grad_mag
    try:
        snake = active_contour(edge_map, snake, alpha=0.01, beta=0.1, gamma=0.001,
                               max_px_move=1.0, max_num_iter=iterations)
    except Exception:
        return mask
    snake_xy = snake[:, ::-1].astype(np.int32)
    new_mask = np.zeros_like(mask, dtype=np.uint8)
    cv2.drawContours(new_mask, [snake_xy], -1, 1, cv2.FILLED)
    return new_mask > 0

# =====================
# Graph Cut refinement
# =====================
def refine_graph_cut(img_gray, mask, iterations=3):
    if np.sum(mask) == 0:
        return mask
    img_color = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    sure_fg = cv2.erode(mask.astype(np.uint8), kernel).astype(bool)
    sure_bg = cv2.dilate((~mask).astype(np.uint8), kernel).astype(bool)
    dist_fg = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 3)
    dist_bg = cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 3)
    gc_mask = np.zeros_like(img_gray, dtype=np.uint8)
    gc_mask[sure_fg] = cv2.GC_FGD
    gc_mask[sure_bg] = cv2.GC_BGD
    thresh = 5.0
    probable_fg = (dist_fg > 0) & (dist_fg < thresh) & (~sure_fg)
    probable_bg = (dist_bg > 0) & (dist_bg < thresh) & (~sure_bg)
    gc_mask[probable_fg] = cv2.GC_PR_FGD
    gc_mask[probable_bg] = cv2.GC_PR_BGD
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(img_color, gc_mask, None, bgd_model, fgd_model, iterations,
                    cv2.GC_INIT_WITH_MASK)
    except Exception:
        return mask
    final_mask = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
    return final_mask > 0

# ===============================
# Chan-Vese level set refinement
# ================================
def refine_chan_vese(img_gray, mask, iterations=150):
    if np.sum(mask) == 0:
        return mask
    try:
        result = morphological_chan_vese(img_gray.astype(np.float32) / 255.0,
                                         iterations=iterations,
                                         init_level_set=mask.astype(bool))
        return result > 0
    except Exception:
        return mask

# =================================================
# Visualization - similarity map overlay (BGR->RGB)
# ==================================================
def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0,2,4))

def blend_overlay(img_gray, mask, hex_color, alpha=0.45, seed_point=None, draw_contour=False, similarity_map=None):
    if mask.shape != img_gray.shape:
        mask = np.zeros_like(img_gray, dtype=bool)
    base = np.stack([img_gray]*3, axis=-1).astype(np.float32)
    if similarity_map is not None and np.any(mask):
        vals = similarity_map[mask]
        sim = 1.0 / (1.0 + vals)
        low, high = np.percentile(sim, [2, 98])
        if high - low > 1e-6:
            scaled = np.zeros_like(similarity_map, dtype=np.float32)
            scaled[mask] = np.clip((sim - low) / (high - low), 0, 1) * 255
        else:
            scaled = np.zeros_like(similarity_map, dtype=np.float32)
        cmap_bgr = cv2.applyColorMap(scaled.astype(np.uint8), cv2.COLORMAP_TURBO)
        cmap = cv2.cvtColor(cmap_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        overlay_color = np.where(mask[..., None], cmap, base/255.0)
        blended = (1-alpha)*base/255.0 + alpha*overlay_color
        result = (blended*255).clip(0,255).astype(np.uint8)
    else:
        rgb = hex_to_rgb(hex_color)
        color_overlay = np.full_like(base, rgb, dtype=np.float32)
        blended = (1-alpha)*base + alpha*color_overlay
        result = np.where(mask[..., None], blended, base)
        result = np.clip(result, 0, 255).astype(np.uint8)
    if seed_point:
        x, y = seed_point
        cv2.circle(result, (int(x), int(y)), 4, (255,255,0), -1)
        cv2.circle(result, (int(x), int(y)), 5, (0,0,0), 1)
    if draw_contour and mask.any():
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(result, contours, -1, (255,0,0), 2)
    return result

def isolate_region(img_gray, mask):
    if mask.shape != img_gray.shape:
        mask = np.zeros_like(img_gray, dtype=bool)
    bg = (img_gray*0.15).astype(np.uint8)
    base = np.stack([bg]*3, -1)
    fg = np.stack([img_gray]*3, -1)
    result = np.where(mask[..., None], fg, base)
    return result.astype(np.uint8)

# ===========
# Analytics
# ===========
def compute_analytics(img_gray, mask):
    pc = int(np.sum(mask))
    if pc==0: return "No region segmented."
    vals = img_gray[mask]
    m = float(np.mean(vals)); sd = float(np.std(vals)); med = float(np.median(vals))
    q1 = float(np.percentile(vals,25)); q3 = float(np.percentile(vals,75))
    mn = int(np.min(vals)); mx = int(np.max(vals))
    hist, _ = np.histogram(vals, bins=256, range=(0, 255), density=True)
    hist = hist[hist > 0]
    entropy = -float(np.sum(hist * np.log2(hist + 1e-12)))
    perc = (np.count_nonzero(img_gray.ravel() <= m) / img_gray.size)*100
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        perim = cv2.arcLength(cnt, True)
        poly_area = cv2.contourArea(cnt)
        circ = (4*np.pi*poly_area)/(perim*perim) if perim>0 else 0
        hull = cv2.convexHull(cnt)
        convex_area = cv2.contourArea(hull)
        solidity = poly_area/convex_area if convex_area>0 else 0
        eq_diam = np.sqrt(4*poly_area/np.pi)
        try:
            (cx,cy),(ma,mi), _ = cv2.fitEllipse(cnt)
            ecc = np.sqrt(1-(min(ma,mi)/max(ma,mi))**2)
            ar = ma/mi if mi!=0 else 0
        except Exception: ecc=0; ar=0
    else: perim=poly_area=circ=solidity=eq_diam=ecc=ar=convex_area=0
    return (
        f"**Metrics**  \n- Pixel Count: {pc}  \n- Poly Area: {poly_area:.0f} px²  \n"
        f"**Intensity**  \n- Mean: {m:.2f}  \n- Median: {med:.2f}  \n- Q1/Q3: {q1:.2f}/{q3:.2f}  \n"
        f"- Min/Max: {mn}/{mx}  \n- Std: {sd:.2f}  \n- Entropy (norm.): {entropy:.2f} bits  \n"
        f"**Shape**  \n- Perimeter: {perim:.1f} px  \n- Circularity: {circ:.3f}  \n"
        f"- Solidity: {solidity:.3f}  \n- Eq.Diameter: {eq_diam:.1f} px  \n"
        f"- Eccentricity: {ecc:.3f}  \n- Aspect Ratio: {ar:.2f}  \n"
        f"**Brightness Percentile**  \nMean intensity percentile: {perc:.1f}"
    )

# =================
# Image preparation
# =================
def _prepare_grayscale(img):
    if img is None: return generate_synthetic_brain()
    if img.ndim==3 and img.shape[-1]==4: img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    if img.ndim==3: img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img = img.astype(np.float32)
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    return img.astype(np.uint8)

# ==============
# Colour palette
# ===============
COLORS = ["Sapphire Blue","Vibrant Amber","Emerald Green","Crimson Red","Magenta","Cyan","Bright Yellow","Orange","Deep Purple"]
COLOR_MAP = {n:h for n,h in zip(COLORS,["#0040FF","#FFBF00","#00C853","#D32F2F","#E040FB","#00BCD4","#FFFF00","#FF6D00","#7C4DFF"])}

def safe_recompute(seed, img_gray, fixed_delta, adaptive_thresh, sensitivity,
                   use_dynamic, use_multi_scale, use_texture, tex_weight,
                   use_grad_dir, grad_dir_weight, grad_weight, dist_weight,
                   edge_factor,
                   apply_clahe, apply_bilateral, apply_aniso,
                   refine_s, snake, show_similarity,
                   use_cost_accept, cost_sensitivity,
                   graph_cut, level_set,
                   use_statistical, stat_threshold,
                   overlay_color, overlay_alpha, draw_contour,
                   progress=None):
    if progress is not None:
        progress(0, desc="Segmenting...")
    if img_gray is None:
        ph = np.zeros((512,512,3),dtype=np.uint8)
        if progress: progress(1, desc="Done")
        return ph, ph, "No image loaded.", np.zeros((512,512), dtype=bool)
    if seed is None:
        if progress: progress(1, desc="Done")
        return (np.stack([img_gray]*3,-1),
                np.zeros((*img_gray.shape,3), dtype=np.uint8),
                "Click to set seed.",
                np.zeros(img_gray.shape, dtype=bool))
    seed = (int(seed[0]), int(seed[1]))
    pp = preprocess_image(img_gray, apply_clahe, apply_bilateral, apply_aniso)
    if refine_s:
        tex_map = fast_local_variance(pp, 5) if use_texture else None
        seed = refine_seed(pp, seed, radius=3, texture_map=tex_map)
    try:
        mask, similarity_map = comprehensive_region_growing(
            img_gray, seed, fixed_delta, adaptive_thresh, sensitivity,
            use_dynamic, use_multi_scale, use_texture, tex_weight,
            use_grad_dir, grad_dir_weight, grad_weight, dist_weight,
            edge_factor=edge_factor,
            use_cost_accept=use_cost_accept, cost_sensitivity=cost_sensitivity,
            use_statistical=use_statistical, stat_threshold=stat_threshold,
            preprocessed_img=pp,
            output_similarity=show_similarity or use_statistical
        )

        if snake:
            mask = refine_boundary_snakes(pp, mask, iterations=None)
        if graph_cut:
            mask = refine_graph_cut(pp, mask, iterations=3)
        if level_set:
            mask = refine_chan_vese(pp, mask, iterations=150)
        # ----------------------------------------------------
        sim_overlay = similarity_map if show_similarity else None
        overlay = blend_overlay(pp, mask, COLOR_MAP[overlay_color], overlay_alpha,
                                seed, draw_contour, similarity_map=sim_overlay)
        isolated = isolate_region(pp, mask)
        report = compute_analytics(img_gray, mask)
        if progress: progress(1, desc="Done")
        return overlay, isolated, report, mask
    except Exception as e:
        traceback.print_exc()
        fb = np.zeros(img_gray.shape, dtype=np.uint8)
        cv2.circle(fb, seed, 5, 1, -1)
        mask = fb > 0
        overlay = blend_overlay(pp, mask, COLOR_MAP[overlay_color], overlay_alpha, seed, False)
        isolated = isolate_region(pp, mask)
        if progress: progress(1, desc="Error")
        return overlay, isolated, f"Error: {str(e)}", mask

# ==============================
# Utility: numpy image to base64
# ===============================
def numpy_to_base64(img):
    if img.dtype != np.uint8:
        img = img.astype(np.uint8)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    pil = Image.fromarray(img)
    buffer = io.BytesIO()
    pil.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()

# ==========
# Dashboard
# ==========
default_synth = generate_synthetic_brain()
default_synth_rgb = np.stack([default_synth]*3, -1)
default_black = np.zeros((*default_synth.shape, 3), dtype=np.uint8)

with gr.Blocks(theme='monochrome', css=custom_css, title="Neuroimaging Dashboard") as demo:
    gr.HTML("<div class='main-header'><h1>Neuroimaging Dashboard</h1><p></p></div>")
    seed_state = gr.State(None)
    current_img = gr.State(default_synth)
    preprocessed_state = gr.State(default_synth)
    mask_state = gr.State(None)
    overlay_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1.2, min_width=380):
            gr.Markdown("## Input")
            upload = gr.Image(type="numpy", label="Upload Brain Slice", height=180)

            with gr.Accordion("Preprocessing", open=True):
                apply_clahe_cb = gr.Checkbox(label="CLAHE", value=False)
                apply_bilateral_cb = gr.Checkbox(label="Bilateral Filter", value=False)
                apply_aniso_cb = gr.Checkbox(label="Anisotropic Diffusion", value=False)

            with gr.Accordion("Algorithm Options", open=True):
                delta_slider = gr.Slider(1,80,30,1,label="Fixed Delta", interactive=True)
                adaptive_cb = gr.Checkbox(label="Adaptive Threshold (MAD)", value=False)
                sensitivity_slider = gr.Slider(1.0,6.0,3.0,0.1,label="Sensitivity", interactive=False)
                dynamic_cb = gr.Checkbox(label="Dynamic Region Mean", value=False)
                multi_scale_cb = gr.Checkbox(label="Multi-scale Gradients", value=False)
                texture_cb = gr.Checkbox(label="Texture Similarity", value=False)
                texture_weight_slider = gr.Slider(0.0,0.5,0.2,0.05,label="Texture Weight", interactive=False)
                grad_dir_cb = gr.Checkbox(label="Gradient Direction Consistency", value=False)
                grad_dir_weight_slider = gr.Slider(0.0,0.5,0.1,0.05,label="Direction Weight", interactive=False)
                grad_weight_slider = gr.Slider(0.0,0.5,0.2,0.05,label="Gradient Cost Weight")
                dist_weight_slider = gr.Slider(0.0,0.1,0.01,0.005,label="Distance Cost Weight")
                edge_factor_slider = gr.Slider(0.2,1.0,0.5,0.05,label="Edge Sensitivity Factor")

            with gr.Accordion("Cost Acceptance", open=False):
                use_cost_accept_cb = gr.Checkbox(label="Multi-cue Cost Acceptance", value=False)
                cost_sensitivity_slider = gr.Slider(0.5, 5.0, 2.0, 0.1, label="Acceptance Strictness", interactive=False)

            with gr.Accordion("Statistical Similarity Model", open=False):
                use_statistical_cb = gr.Checkbox(label="Statistical Similarity (Mahalanobis)", value=False)
                stat_threshold_slider = gr.Slider(1.0, 8.0, 3.0, 0.1, label="Similarity Threshold", interactive=False)

            with gr.Accordion("Post-processing", open=True):
                refine_seed_cb = gr.Checkbox(label="Refine Seed", value=False)
                snake_cb = gr.Checkbox(label="Active Contour", value=False)
                graph_cut_cb = gr.Checkbox(label="Graph Cut", value=False)
                level_set_cb = gr.Checkbox(label="Chan-Vese Level Set", value=False)
                show_similarity_cb = gr.Checkbox(label="Show Similarity Map", value=False)

            with gr.Accordion("Display", open=True):
                overlay_alpha_slider = gr.Slider(0.1,0.9,0.45,0.05,label="Overlay Opacity")
                overlay_color_radio = gr.Radio(choices=COLORS, value=COLORS[0], label="Colour")
                draw_contour_cb = gr.Checkbox(label="Show Contour", value=True)

            with gr.Row():
                reset_btn = gr.Button("Reset All", variant="secondary")
                clear_seed_btn = gr.Button("Clear Seed", variant="secondary")
            with gr.Row():
                export_overlay_btn = gr.Button("Export Overlay (PNG)")
                export_mask_btn = gr.Button("Export Mask (PNG)")
            export_html = gr.HTML(visible=False)

        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column(elem_classes="card"):
                    panel_a = gr.Image(value=default_synth_rgb, type="numpy", label="Interactive Viewport", interactive=True, height=400)
                    gr.Markdown("*Click to set seed point*")
                with gr.Column(elem_classes="card"):
                    panel_b = gr.Image(value=default_black, type="numpy", label="Isolated Region", interactive=False, height=400)
                    gr.Markdown("*Background dimmed to 15%*")

        with gr.Column(scale=0.9, min_width=280):
            gr.HTML("<div class='analytics-header'>Analytics</div>")
            analytics = gr.Markdown(value="Click on the image to set a seed.", elem_classes="analytics-body")

    # ---- Dynamic toggles ---
    adaptive_cb.change(lambda v: gr.update(interactive=v), adaptive_cb, sensitivity_slider)
    adaptive_cb.change(lambda v: gr.update(interactive=not v), adaptive_cb, delta_slider)
    texture_cb.change(lambda v: gr.update(interactive=v), texture_cb, texture_weight_slider)
    grad_dir_cb.change(lambda v: gr.update(interactive=v), grad_dir_cb, grad_dir_weight_slider)
    use_cost_accept_cb.change(lambda v: gr.update(interactive=v), use_cost_accept_cb, cost_sensitivity_slider)
    use_statistical_cb.change(lambda v: gr.update(interactive=v), use_statistical_cb, stat_threshold_slider)

    # ---- Unified on-change handler ----
    def on_any_change(seed, img_gray, delta, adaptive, sensitivity, dynamic, multi_scale, texture, tex_w,
                      grad_dir, dir_w, grad_w, dist_w, edge_factor,
                      use_cost_accept, cost_sensitivity,
                      clahe, bilateral, aniso, refine_s, snake, show_similarity,
                      graph_cut, level_set, use_statistical, stat_threshold,
                      overlay_col, overlay_alpha, draw_contour):
        if img_gray is None:
            return default_synth, default_synth_rgb, default_black, "No image loaded.", None, None
        pp = preprocess_image(img_gray, clahe, bilateral, aniso)
        if seed is None:
            black_img = np.zeros((*pp.shape, 3), dtype=np.uint8)
            return pp, np.stack([pp]*3, -1), black_img, "Click on the image to set a seed.", None, None
        else:
            overlay, isolated, report, mask = safe_recompute(
                seed, img_gray, delta, adaptive, sensitivity,
                dynamic, multi_scale, texture, tex_w, grad_dir, dir_w, grad_w, dist_w,
                edge_factor,
                clahe, bilateral, aniso, refine_s, snake, show_similarity,
                use_cost_accept, cost_sensitivity,
                graph_cut, level_set, use_statistical, stat_threshold,
                overlay_col, overlay_alpha, draw_contour, progress=None)
            return pp, overlay, isolated, report, mask, overlay

    all_controls = [delta_slider, adaptive_cb, sensitivity_slider,
                    dynamic_cb, multi_scale_cb, texture_cb, texture_weight_slider,
                    grad_dir_cb, grad_dir_weight_slider, grad_weight_slider, dist_weight_slider,
                    edge_factor_slider,
                    use_cost_accept_cb, cost_sensitivity_slider,
                    apply_clahe_cb, apply_bilateral_cb, apply_aniso_cb,
                    refine_seed_cb, snake_cb, show_similarity_cb,
                    graph_cut_cb, level_set_cb,
                    use_statistical_cb, stat_threshold_slider,
                    overlay_color_radio, overlay_alpha_slider, draw_contour_cb]
    all_inputs = [seed_state, current_img] + all_controls
    for c in all_controls:
        c.change(on_any_change, inputs=all_inputs,
                 outputs=[preprocessed_state, panel_a, panel_b, analytics, mask_state, overlay_state])

    # ---- Upload handler ----
    def on_upload(image, clahe, bilateral, aniso):
        img_gray = _prepare_grayscale(image)
        pp = preprocess_image(img_gray, clahe, bilateral, aniso)
        return (np.stack([pp]*3,-1), img_gray, pp,
                np.zeros((*img_gray.shape,3),dtype=np.uint8),
                "Image uploaded.", None, None, None)
    upload.change(on_upload,
                  inputs=[upload, apply_clahe_cb, apply_bilateral_cb, apply_aniso_cb],
                  outputs=[panel_a, current_img, preprocessed_state, panel_b, analytics,
                           seed_state, mask_state, overlay_state])

    # ---- Seed click handler ----
    def handle_seed(evt: gr.SelectData, img_gray, preprocessed, delta, adaptive, sensitivity,
                    dynamic, multi_scale, texture, tex_w, grad_dir, dir_w, grad_w, dist_w,
                    edge_factor, use_cost_accept, cost_sensitivity,
                    clahe, bilateral, aniso, refine_s, snake, show_similarity,
                    graph_cut, level_set, use_statistical, stat_threshold,
                    overlay_col, overlay_alpha, draw_contour, progress=gr.Progress()):
        if img_gray is None:
            ph = np.zeros((512,512,3),dtype=np.uint8)
            return (None, ph, ph, "No image loaded.", None, None)
        seed = (int(evt.index[0]), int(evt.index[1]))
        overlay, isolated, report, mask = safe_recompute(
            seed, img_gray, delta, adaptive, sensitivity,
            dynamic, multi_scale, texture, tex_w, grad_dir, dir_w, grad_w, dist_w,
            edge_factor,
            clahe, bilateral, aniso, refine_s, snake, show_similarity,
            use_cost_accept, cost_sensitivity,
            graph_cut, level_set,
            use_statistical, stat_threshold,
            overlay_col, overlay_alpha, draw_contour,
            progress=progress)
        return seed, overlay, isolated, report, mask, overlay

    inputs_seed = [current_img, preprocessed_state, delta_slider, adaptive_cb, sensitivity_slider,
                   dynamic_cb, multi_scale_cb, texture_cb, texture_weight_slider,
                   grad_dir_cb, grad_dir_weight_slider, grad_weight_slider, dist_weight_slider,
                   edge_factor_slider,
                   use_cost_accept_cb, cost_sensitivity_slider,
                   apply_clahe_cb, apply_bilateral_cb, apply_aniso_cb,
                   refine_seed_cb, snake_cb, show_similarity_cb,
                   graph_cut_cb, level_set_cb,
                   use_statistical_cb, stat_threshold_slider,
                   overlay_color_radio, overlay_alpha_slider, draw_contour_cb]
    panel_a.select(handle_seed, inputs=inputs_seed,
                   outputs=[seed_state, panel_a, panel_b, analytics, mask_state, overlay_state])

    # ---- Reset ----
    def reset_all():
        synth = generate_synthetic_brain()
        return (np.stack([synth]*3,-1), synth, synth,
                np.zeros((*synth.shape,3),dtype=np.uint8),
                "Synthetic brain loaded.", 30, False, 3.0, False, False, False, 0.2,
                False, 0.1, 0.2, 0.01, 0.5,
                False, 2.0,
                False, False, False, False, False, False, False, False, False, 3.0,
                0.45, COLORS[0], True, None, None, None)
    reset_btn.click(reset_all, inputs=[], outputs=[panel_a, current_img, preprocessed_state, panel_b, analytics,
                 delta_slider, adaptive_cb, sensitivity_slider,
                 dynamic_cb, multi_scale_cb, texture_cb, texture_weight_slider,
                 grad_dir_cb, grad_dir_weight_slider, grad_weight_slider, dist_weight_slider,
                 edge_factor_slider,
                 use_cost_accept_cb, cost_sensitivity_slider,
                 apply_clahe_cb, apply_bilateral_cb, apply_aniso_cb,
                 refine_seed_cb, snake_cb, show_similarity_cb,
                 graph_cut_cb, level_set_cb,
                 use_statistical_cb, stat_threshold_slider,
                 overlay_alpha_slider, overlay_color_radio, draw_contour_cb,
                 seed_state, mask_state, overlay_state])

    # ---- Clear seed ----
    def clear_seed(preprocessed):
        if preprocessed is None:
            preprocessed = generate_synthetic_brain()
        return (None,
                np.stack([preprocessed]*3,-1),
                np.zeros((*preprocessed.shape,3),dtype=np.uint8),
                "Seed cleared.",
                None,
                None)
    clear_seed_btn.click(clear_seed, inputs=[preprocessed_state],
                         outputs=[seed_state, panel_a, panel_b, analytics, mask_state, overlay_state])

    # ---- Export ----
    def export_overlay(overlay_img):
        if overlay_img is None:
            return gr.update(visible=False), None
        b64 = numpy_to_base64(overlay_img)
        html = f'<a href="data:image/png;base64,{b64}" download="overlay.png">Click to download Overlay</a>'
        return gr.update(visible=True, value=html), overlay_img

    def export_mask(mask_img):
        if mask_img is None:
            return gr.update(visible=False), None
        mask_uint8 = (mask_img.astype(np.uint8)) * 255
        b64 = numpy_to_base64(mask_uint8)
        html = f'<a href="data:image/png;base64,{b64}" download="mask.png">Click to download Mask</a>'
        return gr.update(visible=True, value=html), mask_img

    export_overlay_btn.click(export_overlay, inputs=[overlay_state], outputs=[export_html, overlay_state])
    export_mask_btn.click(export_mask, inputs=[mask_state], outputs=[export_html, mask_state])

demo.launch(debug=True, share=True)